In [11]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
from statsmodels.stats import inter_rater as irr
from sklearn.metrics import f1_score

values_list_eng=['Self-direction','Stimulation','Hedonism','Achievement','Power','Security','Conformity','Tradition','Benevolence','Universalism']

fold='../../../ClassificationByValues/data/extra_data/'
df=pd.read_csv(fold+'extra_validation_experts-gpt-bert.csv', sep="|", encoding ='utf-8')
df.drop('Unnamed: 0', axis=1, inplace=True)


In [3]:
# 1. Aggregate raw experts labels *_e: 1 if the majority is 1
for val in values_list_eng:
    col_e = f"{val}_e"
   
    df[col_e] = (
        df[f"{val}_e_raw"]
        .astype(str)
        .str.split(',')
        .apply(lambda xs: int(sum(1 for x in xs if x.strip() == '1') >= 2)) 
    )

# 2. Aggregate gpt codes *_gpt: 0.6 → 1
for val in values_list_eng:
    col_gpt = f"{val}_gpt"
    df[col_gpt] = (
        df[col_gpt]
        .astype(float)   
        .replace(0.6, 1.0)
        .astype(int)
    )

interrater expert agreement

In [10]:

num_experts = 3  

def parse_triple(s, num_experts=3):
    parts = str(s).split(',')
    labels = []
    for i in range(num_experts):
        if i < len(parts):
            x = parts[i].strip()
            labels.append(int(x) if x in ('0','1') else 0)
        else:
            labels.append(0)
    return labels


expert_rows = []
all_y_true = []
all_y_pred_per_expert = [[] for _ in range(num_experts)]

per_value_metrics = []   


for val in values_list_eng:
    col_raw = f"{val}_e_raw"
    rows_val = df[col_raw].apply(lambda s: parse_triple(s, num_experts)).tolist()
    rows_val_np = np.array(rows_val)  # shape (N, 3)

    # --- 1) for value ---
    dats_exp, cats_exp = irr.aggregate_raters(rows_val)
    fleiss_exp_val = irr.fleiss_kappa(dats_exp, method='fleiss')

    # ---  ---
    per_value_metrics.append({
        "value": val,
        "kappa": round(fleiss_exp_val,3)  
    })

    
    all_y_true.extend(y_true_val.tolist())
    for i in range(num_experts):
        all_y_pred_per_expert[i].extend(rows_val_np[:, i].tolist())

    expert_rows.extend(rows_val)


# ---------- DataFrame ----------
df_metrics_values = pd.DataFrame(per_value_metrics)
print("\n=== Per-value agreement metrics ===")
display(df_metrics_values)


# ---------- fleiss for all df ----------
expert_labels_array = np.array(expert_rows)
dats_exp, cats_exp = irr.aggregate_raters(expert_labels_array)
fleiss_exp_all = irr.fleiss_kappa(dats_exp, method='fleiss')
print(f"\nFleiss' kappa (experts, ALL values): {fleiss_exp_all:.3f}")






=== Per-value agreement metrics ===


,value,kappa
0,Self-direction,0.437
1,Stimulation,0.382
2,Hedonism,0.601
3,Achievement,0.537
4,Power,0.391
5,Security,0.479
6,Conformity,0.265
7,Tradition,0.616
8,Benevolence,0.753
9,Universalism,0.462



Fleiss' kappa (experts, ALL values): 0.597


Domain level

In [7]:
value_domains = {
    "Openness_to_Change": ["Self-direction", "Stimulation", "Hedonism"],
    "Self_Enhancement": ["Achievement", "Power"],
    "Conservation": ["Security", "Conformity", "Tradition"],
    "Self_Transcendence": ["Benevolence", "Universalism"]
}

domains = list(value_domains.keys())
N = len(df)

# ---------- 1.  {value: array (N, 3)} ----------

value_expert_labels = {}

for val in values_list_eng:
    col_raw = f"{val}_e_raw"
    rows_val = df[col_raw].apply(lambda s: parse_triple(s, num_experts)).tolist()
    arr = np.array(rows_val)  # (N, 3)
    value_expert_labels[val] = arr

# ---------- 2.  {domain: array (N, 3)} ----------

domain_expert_labels = {}
maj_domain_matrix = np.zeros((N, len(domains)), dtype=int)

for j, (domain, vals) in enumerate(value_domains.items()):
    # experts: 1, if at least one value from domain = 1
    dom_arr = np.zeros((N, num_experts), dtype=int)
    for val in vals:
        dom_arr = np.maximum(dom_arr, value_expert_labels[val])
    domain_expert_labels[domain] = dom_arr  # (N, 3)

    # majority-domain: max on majority-labels for values in domain
    maj_domain_matrix[:, j] = df[[f"{v}_e" for v in vals]].max(axis=1).values

# ---------- 3. Per-domain kappa,  F1 for experts vs majority ----------

per_domain_metrics = []
domain_rows = []       # 

for j, domain in enumerate(domains):
    dom_arr = domain_expert_labels[domain]        # (N, 3)
    rows_dom = dom_arr.tolist()                   #  [e1,e2,e3] 
    domain_rows.extend(rows_dom)

    # Fleiss' kappa для этого домена
    dats_dom_val, cats_dom_val = irr.aggregate_raters(rows_dom)
    fleiss_dom_val = irr.fleiss_kappa(dats_dom_val, method='fleiss')

    # F1 experts vs majority for the domain
    y_true_dom = maj_domain_matrix[:, j]          # (N,)
    f1_this_domain = []
    for i in range(num_experts):
        y_pred_dom = dom_arr[:, i]
        f1_i = f1_score(
            y_true_dom,
            y_pred_dom,
            pos_label=1,
            average="binary",
            zero_division=0
        )
        f1_this_domain.append(f1_i)
    mean_f1_dom_val = float(np.mean(f1_this_domain))

    per_domain_metrics.append({
        "domain": domain,
        "kappa": round (fleiss_dom_val,3),
        "mean_f1": round(mean_f1_dom_val,3)
    })

# 
df_metrics_domains = pd.DataFrame(per_domain_metrics)
print("\n=== Per-domain agreement metrics ===")
display(df_metrics_domains)

# ---------- 4.  Fleiss' kappa global ----------

expert_labels_domains_array = np.array(domain_rows)  # shape (N*4, 3)
dats_dom_all, cats_dom_all = irr.aggregate_raters(expert_labels_domains_array)
fleiss_dom_all = irr.fleiss_kappa(dats_dom_all, method='fleiss')
print(f"\nFleiss' kappa (experts, ALL domains): {fleiss_dom_all:.3f}")

# ---------- 5. global F1 ----------

y_true_dom_flat = maj_domain_matrix.reshape(-1)

f1_global_dom_list = []
for i in range(num_experts):
    # 
    pred_matrix = np.column_stack(
        [domain_expert_labels[dom][:, i] for dom in domains]
    )  # shape (N, D)

    y_pred_dom_flat = pred_matrix.reshape(-1)

    f1_i = f1_score(
        y_true_dom_flat,
        y_pred_dom_flat,
        pos_label=1,
        average="binary",
        zero_division=0
    )
    f1_global_dom_list.append(f1_i)
    print(f"Expert_{i+1} GLOBAL domain-level F1(1) vs majority: {f1_i:.3f}")

mean_f1_global_dom = float(np.mean(f1_global_dom_list))
print(f"Average expert GLOBAL domain-level F1(1) vs majority: {mean_f1_global_dom:.3f}")



=== Per-domain agreement metrics ===


,domain,kappa,mean_f1
0,Openness_to_Change,0.538,0.836
1,Self_Enhancement,0.523,0.781
2,Conservation,0.499,0.816
3,Self_Transcendence,0.684,0.933



Fleiss' kappa (experts, ALL domains): 0.608
Expert_1 GLOBAL domain-level F1(1) vs majority: 0.861
Expert_2 GLOBAL domain-level F1(1) vs majority: 0.882
Expert_3 GLOBAL domain-level F1(1) vs majority: 0.851
Average expert GLOBAL domain-level F1(1) vs majority: 0.865


GPT-expert agreement

In [5]:
metrics = []

for val in values_list_eng:
    y_true = df[f"{val}_e"]   
    y_pred = df[f"{val}_gpt"]     

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='binary',
        zero_division=0  
    )

    metrics.append({
        "value": val,
        "precision": round(precision,3),
        "recall": round(recall,3),
        "f1": round(f1,3)
    })

metrics_df = pd.DataFrame(metrics)
metrics_df

,value,precision,recall,f1
0,Self-direction,0.284,0.838,0.425
1,Stimulation,0.265,0.686,0.382
2,Hedonism,0.535,0.254,0.344
3,Achievement,0.648,0.654,0.651
4,Power,0.267,0.471,0.340
5,Security,0.784,0.345,0.479
6,Conformity,0.316,0.436,0.366
7,Tradition,0.360,0.525,0.427
8,Benevolence,0.933,0.633,0.754
9,Universalism,0.384,0.458,0.418


In [6]:
#F1 global
from sklearn.metrics import classification_report

y_true_matrix = df[[f"{v}_e" for v in values_list_eng]].values
y_gpt_matrix = df[[f"{v}_gpt" for v in values_list_eng]].values


# print("Accuracy:", accuracy_score(binary_y_val.flatten(), binary_predictions.flatten()))
# print("Classification Report:")
print(classification_report(y_true_matrix.flatten(), y_gpt_matrix.flatten()))

              precision    recall  f1-score   support

           0       0.91      0.92      0.91      8472
           1       0.53      0.53      0.53      1528

    accuracy                           0.86     10000
   macro avg       0.72      0.72      0.72     10000
weighted avg       0.86      0.86      0.86     10000



GPT-expert agreement per domains

In [7]:
results = []

for domain, values in value_domains.items():

    # aggregate to domains
    df[f"{domain}_e"] = df[[f"{v}_e" for v in values]].max(axis=1)
    df[f"{domain}_gpt"] = df[[f"{v}_gpt" for v in values]].max(axis=1)

    y_true = df[f"{domain}_e"]
    y_pred = df[f"{domain}_gpt"]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0
    )

    results.append({
        "domain": domain,
        "precision": round(precision,3),
        "recall": round(recall,3),
        "f1": round(f1,3)
    })

domain_metrics = pd.DataFrame(results)
domain_metrics

,domain,precision,recall,f1
0,Openness_to_Change,0.527,0.699,0.601
1,Self_Enhancement,0.570,0.676,0.618
2,Conservation,0.631,0.446,0.522
3,Self_Transcendence,0.894,0.642,0.747


In [27]:
domain_cols = []
for domain in value_domains.keys():
    domain_cols.extend([
        f"{domain}_e",
        f"{domain}_gpt"
    ])

df_domains = df[['text'] + domain_cols].copy()

y_true_matrix = df_domains[[f"{domain}_e" for domain in value_domains.keys()]].values
y_gpt_matrix = df_domains[[f"{domain}_gpt" for domain in value_domains.keys()]].values

print(classification_report(y_true_matrix.flatten(), y_gpt_matrix.flatten()))

              precision    recall  f1-score   support

           0       0.80      0.84      0.82      2614
           1       0.67      0.61      0.64      1386

    accuracy                           0.76      4000
   macro avg       0.74      0.73      0.73      4000
weighted avg       0.76      0.76      0.76      4000



Critical errors matrix

In [10]:
opposites1 = {
    "Openness_to_Change": "Conservation",
    "Conservation": "Openness_to_Change",
    "Self_Enhancement": "Self_Transcendence",
    "Self_Transcendence": "Self_Enhancement"
}

opposites2 = {
    "Openness_to_Change": "Self_Transcendence",
    "Conservation": "Self_Enhancement",
    "Self_Enhancement": "Conservation",
    "Self_Transcendence": "Openness_to_Change"
}

opposites3 = {
    "Openness_to_Change": "Self_Enhancement",
    "Conservation": "Self_Transcendence",
    "Self_Enhancement": "Openness_to_Change",
    "Self_Transcendence": "Conservation"
}

opposites_list=[opposites1, opposites2, opposites3]

def Critical_Errors(df, true_column_ind, predict_column_ind, domains_dict):

    domains_all = list(value_domains.keys())
    df["critical_error_flag"] = False  # создаём пустой флаг
    results = []

    for d, d_opp in domains_dict.items():
        # 1. 
        expert_has_d      = df[f"{d}{true_column_ind}"] == 1
        expert_no_opp     = df[f"{d_opp}{true_column_ind}"] == 0
        gpt_missed_d      = df[f"{d}{predict_column_ind}"] == 0
        gpt_has_opp       = df[f"{d_opp}{predict_column_ind}"] == 1

        # 2. Check other domains if both experts and gpt agreed
        other_domains = [dom for dom in domains_all if dom not in (d, d_opp)]

        shared_other = pd.Series(False, index=df.index)
        for od in other_domains:
            shared_other |= (df[f"{od}{true_column_ind}"] == 1) & (df[f"{od}{predict_column_ind}"] == 1)

        # 3. criticak error: if no other matching domains!
        critical_mask = (
            expert_has_d &
            expert_no_opp &
            gpt_missed_d &
            gpt_has_opp &
            (~shared_other)
        )
        # 
        df["critical_error_flag"] |= critical_mask

        total_true_cases = expert_has_d.sum()
        critical_errors = critical_mask.sum()
        pct = critical_errors / total_true_cases * 100 if total_true_cases > 0 else 0.0

        results.append({
            "domain_true": d,
            "opposite_predicted": d_opp,
            "expert_cases": total_true_cases,
            "critical_errors": critical_errors,
            "percent_critical": pct
        })

    critical_table = pd.DataFrame(results)
    
    df_critical = df[df["critical_error_flag"]].copy()
    
    return critical_table, df_critical


def AggregateCriticalErrors(df, true_column_ind, predict_column_ind, opposites_list):
    dfc_list=[]
    
    for i in range (0,3):
        dfc_list.append(Critical_Errors(df, true_column_ind, predict_column_ind, opposites_list[i])[0])
    df_all = pd.concat(dfc_list, axis=0).sort_index()
    return df_all

In [11]:
AggregateCriticalErrors(df, '_e', '_gpt', opposites_list)

,domain_true,opposite_predicted,expert_cases,critical_errors,percent_critical
0,Openness_to_Change,Conservation,335,6,1.791045
0,Openness_to_Change,Self_Transcendence,335,5,1.492537
0,Openness_to_Change,Self_Enhancement,335,3,0.895522
1,Conservation,Openness_to_Change,314,31,9.872611
1,Conservation,Self_Enhancement,314,13,4.140127
1,Conservation,Self_Transcendence,314,15,4.777070
2,Self_Enhancement,Self_Transcendence,145,3,2.068966
2,Self_Enhancement,Conservation,145,0,0.000000
2,Self_Enhancement,Openness_to_Change,145,12,8.275862
3,Self_Transcendence,Self_Enhancement,592,11,1.858108


# Prediction by XLM

In [13]:
# load threshold:
import json
with open("../models/xlm-roberta-large/xlm-roberta-large_thresholds_2.json", "r") as f:
    thresholds = json.load(f)

print(thresholds)  

{'Self-direction': 0.324, 'Stimulation': 0.241, 'Hedonism': 0.357, 'Achievement': 0.344, 'Power': 0.441, 'Security': 0.41, 'Conformity': 0.199, 'Tradition': 0.264, 'Benevolence': 0.465, 'Universalism': 0.425, 'GLOBAL': 0.34}


RoBERTa-expert agreement

In [14]:
for val in values_list_eng:
    thr = thresholds[val]  
    prob_col = df[f"{val}_xlm_raw"]     
    df[f"{val}_pred_bin"] = (prob_col >= thr).astype(int)

rows = []

for val in values_list_eng:
    y_true = df[f"{val}_e"].astype(int)          # expert
    y_pred = df[f"{val}_pred_bin"].astype(int)   # xlm

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )

    rows.append({
        "value": val,
        "precision": round(precision,3),
        "recall": round(recall,3),
        "f1": round(f1,3)
    })

metrics_df = pd.DataFrame(rows)
print ("xlm vs experts")
metrics_df

xlm vs experts


,value,precision,recall,f1
0,Self-direction,0.264,0.829,0.400
1,Stimulation,0.230,0.614,0.335
2,Hedonism,0.521,0.292,0.374
3,Achievement,0.627,0.583,0.604
4,Power,0.167,0.265,0.205
5,Security,0.833,0.258,0.394
6,Conformity,0.208,0.273,0.236
7,Tradition,0.429,0.541,0.478
8,Benevolence,0.910,0.675,0.775
9,Universalism,0.355,0.325,0.340


In [15]:
# Global F1 for XLM
y_true_matrix = df[[f"{v}_e" for v in values_list_eng]].values
y_xlm_matrix = df[[f"{v}_pred_bin" for v in values_list_eng]].values

print(classification_report(y_true_matrix.flatten(), y_xlm_matrix.flatten()))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91      8472
           1       0.51      0.51      0.51      1528

    accuracy                           0.85     10000
   macro avg       0.71      0.71      0.71     10000
weighted avg       0.85      0.85      0.85     10000



RoBERTa-expert agreement, domain level

In [16]:
results = []

for domain, values in value_domains.items():

    df[f"{domain}_xlm"] = df[[f"{v}_pred_bin" for v in values]].max(axis=1)

    y_true = df[f"{domain}_e"]
    y_pred = df[f"{domain}_xlm"]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0
    )

    results.append({
        "domain": domain,
        "precision": round(precision,3),
        "recall": round(recall,3),
        "f1": round(f1,3)
    })

domain_metrics = pd.DataFrame(results)
domain_metrics

,domain,precision,recall,f1
0,Openness_to_Change,0.501,0.710,0.588
1,Self_Enhancement,0.535,0.586,0.559
2,Conservation,0.659,0.382,0.484
3,Self_Transcendence,0.902,0.667,0.767


In [17]:
# F1 global:

y_true_matrix = df[[f"{domain}_e" for domain in value_domains.keys()]].values
y_xlm_matrix = df[[f"{domain}_xlm" for domain in value_domains.keys()]].values

print(classification_report(y_true_matrix.flatten(), y_xlm_matrix.flatten()))

              precision    recall  f1-score   support

           0       0.80      0.84      0.82      2614
           1       0.67      0.60      0.63      1386

    accuracy                           0.76      4000
   macro avg       0.73      0.72      0.73      4000
weighted avg       0.75      0.76      0.76      4000



Critical errors for RoBERTa

In [18]:
AggregateCriticalErrors(df, '_e', '_xlm', opposites_list)

,domain_true,opposite_predicted,expert_cases,critical_errors,percent_critical
0,Openness_to_Change,Conservation,335,5,1.492537
0,Openness_to_Change,Self_Transcendence,335,5,1.492537
0,Openness_to_Change,Self_Enhancement,335,3,0.895522
1,Conservation,Openness_to_Change,314,49,15.605096
1,Conservation,Self_Enhancement,314,13,4.140127
1,Conservation,Self_Transcendence,314,9,2.866242
2,Self_Enhancement,Self_Transcendence,145,0,0.000000
2,Self_Enhancement,Conservation,145,1,0.689655
2,Self_Enhancement,Openness_to_Change,145,11,7.586207
3,Self_Transcendence,Self_Enhancement,592,3,0.506757


Spearmen for XLM and experts

In [54]:
from scipy.stats import spearmanr


prob_map = {
    0: 0.0,
    1: 0.3,
    2: 0.6,
    3: 1.0
}

def count_ones(triple_str: str) -> int:
    """
    triple_str: строка вида "0,1,0" или "1,1,1".
    Возвращает количество '1' (0..3).
    """
    parts = str(triple_str).split(',')
    return sum(1 for x in parts if x.strip() == '1')

# создаём вероятностные экспертные колонки: <Value>_e_prob
for val in values_list_eng:
    raw_col = f"{val}_e_raw"
    prob_col = f"{val}_e_prob"

    df[prob_col] = (
        df[raw_col]
        .apply(count_ones)
        .map(prob_map)
    )
    
    # 2. Считаем Spearman per-value: модельные вероятности vs экспертные вероятности
rows = []

for val in values_list_eng:
    expert_col = f"{val}_e_prob"  # 0, 0.3, 0.6, 1
    model_col = f"{val}_xlm_raw"              # float [0,1] из модели

    # маска валидных значений (на случай NaN)
    mask = df[[expert_col, model_col]].notna().all(axis=1)

    if mask.sum() < 3:
        # слишком мало наблюдений — Spearman не имеет смысла
        rho, p = np.nan, np.nan
    else:
        rho, p = spearmanr(df.loc[mask, expert_col], df.loc[mask, model_col])

    rows.append({
        "value": val,
        "n": int(mask.sum()),
        "spearman_rho": round(rho,3),
        "p_value": p,
    })

spearman_by_value = pd.DataFrame(rows)
print(spearman_by_value)

# 3. Macro-average Spearman (средний ρ по ценностям)
macro_rho = spearman_by_value["spearman_rho"].mean()
print(f"\nMacro-average Spearman rho (model vs experts, across 10 values): {macro_rho:.3f}")

# 4. Global Spearman: все ценности и все посты в один вектор (flatten)
expert_all = []
model_all = []

for val in values_list_eng:
    expert_col = f"{val}_e_prob"
    model_col = f"{val}_xlm_raw" 
    mask = df[[expert_col, model_col]].notna().all(axis=1)

    expert_all.append(df.loc[mask, expert_col].values)
    model_all.append(df.loc[mask, model_col].values)

expert_all = np.concatenate(expert_all)
model_all = np.concatenate(model_all)

rho_global, p_global = spearmanr(expert_all, model_all)
print(f"\nGlobal Spearman rho (flattened across all values and posts): {rho_global:.3f}, p = {p_global:.3e}")

            value     n  spearman_rho        p_value
0  Self-direction  1000         0.494   1.169775e-62
1     Stimulation  1000         0.394   1.482766e-38
2        Hedonism  1000         0.401   5.106409e-40
3     Achievement  1000         0.553   3.042874e-81
4           Power  1000         0.327   2.384694e-26
5        Security  1000         0.507   1.622562e-66
6      Conformity  1000         0.367   2.817808e-33
7       Tradition  1000         0.356   3.078312e-31
8     Benevolence  1000         0.745  8.076802e-178
9    Universalism  1000         0.386   6.537169e-37

Macro-average Spearman rho (model vs experts, across 10 values): 0.453

Global Spearman rho (flattened across all values and posts): 0.490, p = 0.000e+00


RoBERTa-GPT agreement

In [19]:
y_true_matrix = df[[f"{v}_gpt" for v in values_list_eng]].values
y_xlm_matrix = df[[f"{v}_pred_bin" for v in values_list_eng]].values

print(classification_report(y_true_matrix.flatten(), y_xlm_matrix.flatten()))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95      8473
           1       0.71      0.70      0.71      1527

    accuracy                           0.91     10000
   macro avg       0.83      0.83      0.83     10000
weighted avg       0.91      0.91      0.91     10000

